# CCTV Weapon Detection Dataset: quick start

Free sample of the synthetic **CCTV Weapon Detection Dataset** by [Simuletic](https://simuletic.com/?utm_source=github&utm_medium=notebook&utm_campaign=cctv-weapon-dataset): 60 CCTV images with YOLO and COCO boxes for `person`, `handgun`, `rifle`, `shotgun`, `sniper_rifle`.

This notebook: 1. downloads the sample, 2. counts and draws the labels, 3. fine-tunes YOLO11 for a few epochs, 4. runs inference on the validation images.

In Colab, choose **Runtime → Change runtime type → T4 GPU** first. Training 20 epochs takes a few minutes.

In [ ]:
%pip install -q ultralytics

## 1. Get the data

Clones the repo (images, labels, `data.yaml`). If you run this notebook inside a clone, it uses that.

In [ ]:
import os
from pathlib import Path

if Path("data.yaml").exists():
    ROOT = Path(".").resolve()
elif Path("../data.yaml").exists():
    ROOT = Path("..").resolve()
else:
    if not Path("cctv-weapon-dataset").exists():
        !git clone --depth 1 https://github.com/Simuletic/cctv-weapon-dataset
    ROOT = Path("cctv-weapon-dataset").resolve()
print(ROOT)

## 2. Look at the labels

YOLO labels are one `.txt` per image: `class x_center y_center width height`, normalised to 0–1. COCO versions are in `annotations/`.

In [ ]:
import yaml
from collections import Counter

cfg = yaml.safe_load((ROOT / "data.yaml").read_text())
names = cfg["names"]

def read_labels(img):
    p = ROOT / "labels" / img.parent.name / (img.stem + ".txt")
    return [(int(a[0]), *map(float, a[1:5])) for a in (l.split() for l in p.read_text().splitlines()) if a]

for split in ("train", "val"):
    imgs = sorted((ROOT / "images" / split).glob("*.jpg"))
    boxes = Counter(names[c] for i in imgs for c, *_ in read_labels(i))
    neg = sum(1 for i in imgs if not {names[c] for c, *_ in read_labels(i)} - {"person"})
    print(f"{split}: {len(imgs)} images, {neg} hard negatives, boxes {dict(boxes)}")

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

colors = ["#3ca0ff", "#ff4646", "#ff8c00", "#ff3cc8", "#be5aff"]
imgs = random.Random(1).sample(sorted((ROOT / "images" / "train").glob("*.jpg")), 6)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, img in zip(axes.flat, imgs):
    im = Image.open(img)
    w, h = im.size
    ax.imshow(im)
    for c, x, y, bw, bh in read_labels(img):
        ax.add_patch(patches.Rectangle(((x - bw / 2) * w, (y - bh / 2) * h), bw * w, bh * h,
                                       fill=False, color=colors[c % 5], lw=2))
        if names[c] != "person":
            ax.text((x - bw / 2) * w, (y - bh / 2) * h - 8, names[c], color="white", fontsize=9,
                    backgroundcolor=colors[c % 5])
    ax.set_title(img.name)
    ax.axis("off")
plt.tight_layout()

## 3. Fine-tune YOLO11

`data.yaml` has no `path:` entry, so Ultralytics resolves the splits relative to the file. We pass an absolute copy anyway so the notebook works from any working directory.

In [ ]:
from ultralytics import YOLO

cfg_abs = dict(cfg, path=str(ROOT))
(ROOT / "data_abs.yaml").write_text(yaml.safe_dump(cfg_abs, sort_keys=False))

model = YOLO("yolo11n.pt")  # try yolo11s.pt for better accuracy
model.train(data=str(ROOT / "data_abs.yaml"), epochs=20, imgsz=640, batch=8,
            project=str(ROOT / "runs"), name="train", exist_ok=True)

In [ ]:
metrics = model.val(data=str(ROOT / "data_abs.yaml"), imgsz=640, project=str(ROOT / "runs"), name="val", exist_ok=True)
print(f"mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}")

Expect low numbers: 48 training images are enough to check that your training code works, not to train a detector. The [full dataset](https://simuletic.com/datasets/cctv-weapon-detection-dataset?utm_source=github&utm_medium=notebook&utm_campaign=cctv-weapon-dataset) has 1,000 images with the same layout; point `data=` at its `data.yaml`.

## 4. Run inference

In [ ]:
results = model.predict(source=str(ROOT / "images" / "val"), imgsz=640, conf=0.25, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, results[:6]):
    ax.imshow(r.plot()[:, :, ::-1])
    ax.set_title(Path(r.path).name)
    ax.axis("off")
plt.tight_layout()

## Next steps

- Full **CCTV Weapon Detection Dataset**: 1,000 images, same labels and layout, research or commercial licence: [https://simuletic.com/datasets/cctv-weapon-detection-dataset](https://simuletic.com/datasets/cctv-weapon-detection-dataset?utm_source=github&utm_medium=notebook&utm_campaign=cctv-weapon-dataset)
- Other scenes, cameras or classes: [custom data](https://simuletic.com/custom-data?utm_source=github&utm_medium=notebook&utm_campaign=cctv-weapon-dataset)
- Sample licence: CC BY-NC 4.0. Questions: contact@simuletic.com